<div align="center">
  <img src="https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/wsu_logo_horizontal.png" alt="Wayne State University Logo" width="320">
  <h1>Homework 3: Tool Wear Classification and Feature Choice</h1>
</div>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WSU-AI-in-ME/ai-in-me-1/blob/main/homework/hw03_tool_wear_estimation_and_feature_comparison.ipynb)

[Course Repository](https://github.com/WSU-AI-in-ME/ai-in-me-1) · [Homework Index](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/Homework_index.ipynb)

### ME 5995 — AI in Mechanical Engineering I: Fundamentals of Manufacturing Data Science
**Wayne State University · Individual Homework · Approximately 45–60 minutes**

**Version: Student Notebook**

Open in Colab and save a personal copy before editing. Canvas controls
the official due date and submission requirements.

## Student Information

Double-click this Markdown cell to edit.

**Student name:** TODO: Enter your name  
**WSU AccessID:** TODO: Enter your WSU AccessID

## Overview

Can changing three sensor features change Severe-wear detection? Choose three
features for model B, run the supplied Logistic Regression example A, then fit B
and compare their results using the same training/validation rows and model settings.

**Your work:** one feature list, three code expressions (fit, predict, F1), one
comparison table and three short sentences. Use Lab 5 as a code reference. No raw-signal
processing, regression, model tuning or repeated feature search is required.

Run cells in order. Replace each required `None` with your expression, preserving
variable names and indentation. After an edit, rerun that cell and those below it.
Provided setup/reporting runs unchanged. An unchanged starter is not completed work.
The 45–60 minute estimate is a planning target, not measured student timing.

## Data and setup

Use Google Colab; the supplied code downloads the course feature CSV. Each row
is one cut from the PHM 2010 dataset. We reuse Lab 5's c1/c4 split: 504 training
and 126 validation cuts. c6 is not used. Nearby cuts can be correlated; these
validation results do not establish performance on another cutter.

**Target:** Severe = 1 when mean flute wear is at least 150 µm; Normal = 0 otherwise.
These course-defined labels are not industrial replacement limits.
[Feature definitions and provenance](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/phm2010/features/README.md).

In [ ]:
# PROVIDED — load the tools and course table. No data upload is needed.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix

data_url = 'https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/data/phm2010/features/phm2010_features.csv'
data = pd.read_csv(data_url)
# Reject an older label definition; do not silently relabel the source.
expected_level = pd.Series(0, index=data.index)
expected_level.loc[data['wear_mean_um'] > 75] = 1
expected_level.loc[data['wear_mean_um'] >= 150] = 2
if not np.array_equal(expected_level, data['wear_level']):
    raise ValueError('Download the current course CSV with 75/150 micrometer boundaries.')

# Keep c6 out of this assignment. One fixed split is shared by A and B.
development = data.loc[data['cutter_id'].isin(['c1', 'c4'])].copy()
strata = development['cutter_id'] + '_' + development['wear_level'].astype(str)
train, valid = train_test_split(development, test_size=0.20, random_state=42, stratify=strata)
train = train.sort_index()
valid = valid.sort_index()
# True becomes 1 (Severe), False becomes 0 (Normal).
y_train_severe = (train['wear_level'] == 2).astype(int)
y_valid_severe = (valid['wear_level'] == 2).astype(int)
print('Training / validation cuts:', len(train), len(valid))

## 1. Choose your features (25 points)

Group A is fixed: `force_y_rms`, `vibration_y_rms`, `ae_rms_rms`.
Choose **three distinct columns** for B from the nine below. At least one must
differ from A; reordering A is not a new group. You may mix statistics and channels.

| Channel (original units) | RMS | Standard deviation | Peak-to-peak |
|---|---|---|---|
| Force Y (N) | `force_y_rms` | `force_y_sd` | `force_y_peak_to_peak` |
| Vibration Y (g) | `vibration_y_rms` | `vibration_y_sd` | `vibration_y_peak_to_peak` |
| AE-RMS (V) | `ae_rms_rms` | `ae_rms_sd` | `ae_rms_peak_to_peak` |

RMS summarizes overall magnitude, including a nonzero mean. Standard deviation
summarizes variation around the mean. Peak-to-peak is maximum minus minimum and
can be affected by extremes. All retain the original channel's units before scaling.
AE-RMS is already a processed channel; its features do not describe raw AE waveforms.

Write a Python list of quoted column names, separated by commas inside `[...]`.
For example, a two-column list has the form `['force_y_sd', 'vibration_y_sd']`;
your list must contain three columns. Select only from the table: wear labels,
wear measurements, cutter ID and cut number cannot be inputs.

### Use training correlations as a guide

Run the supplied cell; no correlation code needs editing. The table compares each
feature with **continuous mean wear**, not the binary label. Pearson r ranges from
-1 to 1: its sign gives the direction and its magnitude indicates linear association.
Near zero does not rule out a nonlinear relationship. A large absolute r does not
guarantee better Severe F1 or establish causation.

The heatmap compares features with each other. A large absolute r can suggest
overlapping information, but does not require dropping either feature. Ignore the
diagonal (each feature correlates with itself). Values are rounded: an off-diagonal
1.00 does not necessarily mean two features are identical. Use these results with the
signal definitions above; there is no required correlation cutoff or best group.
Only training rows are used. Choose B before fitting it. You may correct code or
column-name errors; do not try new feature groups to improve the validation score.

In [ ]:
# PROVIDED — Pearson correlations use training rows only, never validation rows.
candidate_features = ['force_y_rms', 'force_y_sd', 'force_y_peak_to_peak',
    'vibration_y_rms', 'vibration_y_sd', 'vibration_y_peak_to_peak',
    'ae_rms_rms', 'ae_rms_sd', 'ae_rms_peak_to_peak']
training_features = train[candidate_features]
# Match each feature to the continuous wear measurement for the same training cut.
wear_correlations = training_features.corrwith(train['wear_mean_um'], method='pearson')
correlation_table = wear_correlations.to_frame(name='Pearson r with mean wear')
display(correlation_table.round(3))  # Round for reading; retain the original values.

# Compare every pair of feature columns using the same training rows.
feature_correlations = training_features.corr(method='pearson')
# Short labels keep the plot readable; use the table above to identify full names.
short_names = ['Force RMS', 'Force SD', 'Force P-P', 'Vib RMS', 'Vib SD',
               'Vib P-P', 'AE RMS', 'AE SD', 'AE P-P']
fig, ax = plt.subplots(figsize=(9, 7))
heatmap = ax.imshow(feature_correlations, vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(range(9), short_names, rotation=45, ha='right')
ax.set_yticks(range(9), short_names)
# Supplied plotting loops place the numeric coefficients in the cells; do not edit.
for row in range(9):
    for column in range(9):
        value = feature_correlations.iloc[row, column]
        ax.text(column, row, f'{value:.2f}', ha='center', va='center',
                color='white' if abs(value) > 0.6 else 'black', fontsize=8)
fig.colorbar(heatmap, ax=ax, label='Pearson r (unitless)')
ax.set_title('Feature relationships — training cuts only (P-P = peak-to-peak)')
fig.tight_layout()
plt.show()

In [ ]:
# REQUIRED — replace None with your three selected column names.
features_b = None  # TODO: Three distinct allowed features, not the same set as A.

**Selection reasons — before fitting B (two short sentences):**

1. **Statistical:** cite one Pearson r linking a selected feature to mean wear
   (table), or linking two selected features (heatmap), and explain how it informed
   your choice. Either relationship is sufficient; one numeric citation is enough.
2. **Signal/physical:** explain what information a selected feature represents
   (such as magnitude, variability or range). Do not claim an unverified cause.

**Statistical:** TODO

**Signal/physical:** TODO

In [ ]:
# PROVIDED — check the list before using it to select columns.
features_a = ['force_y_rms', 'vibration_y_rms', 'ae_rms_rms']
allowed_features = ['force_y_rms', 'force_y_sd', 'force_y_peak_to_peak',
    'vibration_y_rms', 'vibration_y_sd', 'vibration_y_peak_to_peak',
    'ae_rms_rms', 'ae_rms_sd', 'ae_rms_peak_to_peak']
features_ready = False
if features_b is not None:
    if not isinstance(features_b, list) or len(features_b) != 3:
        raise ValueError('Use a Python list containing exactly three column names.')
    if not pd.Index(features_b).isin(allowed_features).all():
        raise ValueError('Choose only column names from the allowed table.')
    if len(set(features_b)) != 3 or set(features_b) == set(features_a):
        raise ValueError('Choose three distinct features, with at least one different from A.')
    features_ready = True
else:
    print('Complete the feature list before fitting B.')

# A uses its own training means and standard deviations.
X_train_a = train[features_a]
X_valid_a = valid[features_a]
scaler_a = StandardScaler()
scaler_a.fit(X_train_a)  # Learn statistics only from training rows.
X_train_a_scaled = scaler_a.transform(X_train_a)
X_valid_a_scaled = scaler_a.transform(X_valid_a)  # Reuse those statistics.
if features_ready:
    # Select the same columns in the same order for training and validation.
    X_train_b = train[features_b]
    X_valid_b = valid[features_b]
    scaler_b = StandardScaler()  # B needs separate training statistics.
    scaler_b.fit(X_train_b)
    X_train_b_scaled = scaler_b.transform(X_train_b)
    X_valid_b_scaled = scaler_b.transform(X_valid_b)
    print('B training / validation shapes:', X_train_b_scaled.shape, X_valid_b_scaled.shape)

## 2. Fit and evaluate B (45 points)

The completed A example below shows the required workflow. Adapt it for B using
the variable names in the table; keep model settings and the default threshold unchanged.

| Purpose | B variable |
|---|---|
| Model | `class_model_b` |
| Scaled training / validation inputs | `X_train_b_scaled` / `X_valid_b_scaled` |
| Training labels / validation labels | `y_train_severe` / `y_valid_severe` |
| Fitted model / predictions / score | `fit_b` / `pred_b` / `f1_b` |

`fit(X, y)` learns from training rows; `predict(X)` produces one label per validation
row. Use validation labels only to evaluate predictions. `fit_b` is a model object,
`pred_b` contains 126 labels, and `f1_b` is one number. In `f1_score(actual, predicted)`,
Severe (1) is positive; `zero_division=0` reports zero for an undefined ratio.

In [ ]:
# COMPLETED EXAMPLE — reference model A, using fixed RMS features.
class_model_a = LogisticRegression(C=1.0, solver='lbfgs', max_iter=2000)
fit_a = class_model_a.fit(X_train_a_scaled, y_train_severe)
pred_a = class_model_a.predict(X_valid_a_scaled)
f1_a = f1_score(y_valid_severe, pred_a, zero_division=0)
print('Reference A validation Severe F1:', round(f1_a, 3))

In [ ]:
# PROVIDED initialization — do not edit these three lines.
fit_b = None
pred_b = None
f1_b = None
if features_ready:
    class_model_b = LogisticRegression(C=1.0, solver='lbfgs', max_iter=2000)
    fit_b = None  # TODO: Fit B on its scaled training inputs and training labels.
    if fit_b is not None:
        pred_b = None  # TODO: Predict labels for B's scaled validation inputs.
        if pred_b is not None:
            f1_b = None  # TODO: Score predictions against actual validation labels.

## 3. Compare the results (30 points)

Run the supplied report, then fill the table and write **one sentence** naming the
higher-F1 group (or a tie) and the accompanying FP/FN difference. Higher F1 is better
for this comparison; improvement is not required for credit.

FP is actual Normal predicted Severe; FN is actual Severe predicted Normal.
The split, model settings, scaling procedure and target stay fixed, so this compares
feature groups on these validation cuts.

In [ ]:
# PROVIDED — report metrics only when the required B work is present.
if f1_b is not None:
    matrix_a = confusion_matrix(y_valid_severe, pred_a, labels=[0, 1])
    matrix_b = confusion_matrix(y_valid_severe, pred_b, labels=[0, 1])
    # Rows are actual, columns are predicted: [0, 1] is FP; [1, 0] is FN.
    comparison = pd.DataFrame()
    comparison['Group'] = ['A', 'B']
    comparison['Severe F1'] = [f1_a, f1_b]
    comparison['FP'] = [matrix_a[0, 1], matrix_b[0, 1]]
    comparison['FN'] = [matrix_a[1, 0], matrix_b[1, 0]]
    display(comparison.round(3))  # Rounding affects display, not the model.
else:
    print('Complete the feature list and all three B expressions, then rerun the report.')

| Group | Severe F1 (3 decimals) | FP | FN |
|---|---:|---:|---:|
| A | TODO | TODO | TODO |
| B | TODO | TODO | TODO |

**Comparison (one sentence):** TODO

## Submission checklist

Rendered checkboxes are not clickable; edit `[ ]` to `[x]`.

- [ ] Enter name and WSU AccessID.
- [ ] Complete one feature list, three model/metric expressions, the table and three sentences.
- [ ] Restart and Run all; retain the correlation table, heatmap, A score and A/B comparison outputs.
- [ ] Save `HW03_Firstname_Lastname.ipynb` and a matching readable `.pdf`; submit both through Canvas.

No separate report. PDF export defects receive corrective feedback. Canvas controls
the due date.

## References

- [PHM Society 2010 Data Challenge](https://phmsociety.org/phm_competition/2010-phm-society-conference-data-challenge/)
- [Course feature data dictionary](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/phm2010/features/DATA_DICTIONARY.md)
- Instructor-provided Lab 5: scaling, Logistic Regression and Severe F1 examples.

The course-derived processing and labels do not replace the original dataset license.